In [11]:
from typing import Any
from fastapi import FastAPI,Response
from pydantic import BaseModel,EmailStr
from fastapi.responses import JSONResponse,RedirectResponse
app = FastAPI()


In [2]:
# 返回类型

class Item(BaseModel):
    name :str
    description: str | None = None
    price: float 
    tax : float | None = None 
    tags: list[str] = []



@app.post("/items/")
def create_item(item:Item) ->Item:
    return item


@app.post("/items/")
def read_items() ->list[Item]:
    return [
        Item(name="Protal Gun",price =42.0),
        Item(name = "Plumbus",price=32.0)
    ]



In [6]:
@app.post("/items/", response_model=Item)
async def create_item(item: Item) -> Any:
    return item


@app.get("/items/", response_model=list[Item])
async def read_items() -> Any:
    return [
        {"name": "Portal Gun", "price": 42.0},
        {"name": "Plumbus", "price": 32.0},
    ]

In [7]:
# 使用EmailStr
class UserIn(BaseModel):
    username: str
    password: str
    email: EmailStr
    full_name: str | None = None

@app.post("/user/")
def create_user(user: UserIn) ->UserIn:
    return user

In [8]:
# 输出模型

class UserOut(BaseModel):
    username: str
    email: EmailStr
    full_name: str | None = None

@app.post("/user/",response_model=UserOut)
def create_user(user:UserIn)->Any:
    return user
    

In [10]:
# 减少重复字段，采用继承

class BaseUser(BaseModel):
    username: str
    email: EmailStr
    full_name: str | None = None

class UserIn2(BaseUser):
    password: str


@app.get("/user/")
def create_user(user:UserIn2)-> BaseUser:
    return user

In [12]:
# fastapi 自带的response
@app.get("/portal")
def get_portal(teleport: bool = False) ->Response:
    if teleport:
        return RedirectResponse(url="http://")
    return JSONResponse(content={"message":"Here's your interdimensional portal."})


In [13]:
# response_model_exclude_unset=True 排除默认值

class Item(BaseModel):
    name: str
    description: str | None = None
    price: float
    tax: float = 10.5
    tags: list[str] = []


items = {
    "foo": {"name": "Foo", "price": 50.2},
    "bar": {"name": "Bar", "description": "The bartenders", "price": 62, "tax": 20.2},
    "baz": {"name": "Baz", "description": None, "price": 50.2, "tax": 10.5, "tags": []},
}



@app.get("/items/{item_id}", response_model=Item, response_model_exclude_unset=True)
async def read_item(item_id: str):
    return items[item_id]

In [14]:

@app.get(
    "/items/{item_id}/name",
    response_model=Item,
    response_model_include={"name", "description"},
)
async def read_item_name(item_id: str):
    return items[item_id]


@app.get("/items/{item_id}/public", response_model=Item, response_model_exclude={"tax"})
async def read_item_public_data(item_id: str):
    return items[item_id]